# Build a RAG chatbot in 90 minutes

We will build a chatbot that answers questions about three scientific papers and shows
you the exact page each answer comes from. It runs in your browser, and you watch it grow
while you work through this notebook.

![The two halves of RAG: preparing the documents once, and answering every question](img/overview.svg)

**How this works.** Some cells start with `%%writefile app.py`. They don't run code;
they *write* the chatbot's source file. A Chainlit server watches that file and reloads
whenever it changes. So: run the cell, switch to the browser tab, and the bot has a new skill.

![A notebook cell writes app.py, Chainlit notices and reloads the browser](img/live-reload.svg)

| Part | What you learn | The bot after it |
|---|---|---|
| 0 | Setup | Repeats what you say |
| 1 | Talking to an LLM | A plain chatbot that makes things up |
| 2 | Reading PDFs and cutting them into chunks | — |
| 3 | Embeddings and Qdrant | A search engine over the papers |
| 4 | Retrieval-augmented generation | Answers from the papers |
| 5 | Citations | Answers that link to the page |
| 6 | Where to go from here | — |

Run the cells from top to bottom with **Shift+Enter**.

## Part 0 · Setup

The cell below reads your gateway address and key from the `.env` file and asks the
model for one word. If it prints something, you're ready.

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBED_MODEL = os.environ["EMBED_MODEL"]
llm = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

reply = llm.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Say hello in one word."}],
)
print(reply.choices[0].message.content)

Hello.


### The first version of the bot

[Chainlit](https://docs.chainlit.io) turns a Python function into a chat window.
Whatever function carries `@cl.on_message` is called with every message you type.

In [2]:
%%writefile app.py
import chainlit as cl


@cl.on_message
async def on_message(message: cl.Message):
    await cl.Message(content=f"You said: {message.content}").send()

Writing app.py


Now start the server. Run this cell **once**: the server keeps running in the
background and reloads by itself whenever `app.py` changes. Its messages go to
`chainlit.log`.

Then open **http://localhost:8000** in your browser and say something.

In [3]:
import subprocess
import sys

server = subprocess.Popen(
    [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
    stdout=open("chainlit.log", "w"),
    stderr=subprocess.STDOUT,
)
print("Open http://localhost:8000")

Open http://localhost:8000


> **If the bot misbehaves later**, look at the end of the log: `!tail -20 chainlit.log`.
> If the browser can't connect after a kernel restart, run the cell above again.

## Part 1 · Talking to an LLM

An LLM call is a list of messages in, one message out. Let's ask about a detail from one
of our three papers.

In [4]:
QUESTION = "In Kage et al. (2018), what were the lifetime-encoded beads loaded with, and which lifetimes did they have?"

reply = llm.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": QUESTION}],
)
print(reply.choices[0].message.content)

In Kage et al. (2018), the lifetime-encoded beads were loaded with **quantum dots (QDs)**.

The beads were loaded with QDs possessing the following three distinct fluorescence lifetimes:

*   **~2 ns**
*   **~10 ns**
*   **~25 ns**

These beads were used as internal calibration standards to enable the quantitative mapping of fluorescence lifetimes in biological samples.


It sounds confident. Is it right? The model has never seen the paper, so the answer is
either general knowledge or invented. We'll come back to this question in Part 4.

The bot gets the same ability now. Two additions compared to the notebook:
- **History.** The model has no memory, so we send the whole conversation every time.
  `cl.user_session` keeps it per browser tab.
- **Streaming.** `stream=True` delivers the answer token by token, so you see it being typed.
  The app uses `AsyncOpenAI`, the same client in a form Chainlit can run without blocking.

In [5]:
%%writefile app.py
import os

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
SYSTEM_PROMPT = "You are a helpful assistant. Keep your answers short."


@cl.on_chat_start
async def on_chat_start():
    cl.user_session.set("history", [{"role": "system", "content": SYSTEM_PROMPT}])


@cl.on_message
async def on_message(message: cl.Message):
    history = cl.user_session.get("history")
    history.append({"role": "user", "content": message.content})

    answer = cl.Message(content="")
    stream = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"], messages=history, stream=True
    )
    async for part in stream:
        if part.choices and part.choices[0].delta.content:
            await answer.stream_token(part.choices[0].delta.content)
    await answer.send()

    history.append({"role": "assistant", "content": answer.content})

Overwriting app.py


Switch to the browser. The page has reloaded; ask it the same question about Kage et al.

## Part 2 · Reading PDFs and cutting them into chunks

To answer from the papers, the bot first has to read them. A PDF only stores where to draw
each letter, not what is a heading, a paragraph or a table.
[Docling](https://github.com/docling-project/docling) looks at each page with a layout model,
works out its structure and writes it out as Markdown.

![Docling finds headings, tables and figures on a PDF page and writes Markdown](img/docling.svg)

We convert page by page and keep the file name and page number with every page; we'll need
them for the citations in Part 5. This takes about a minute.

In [6]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# These papers contain real text, so we skip OCR (reading text out of images).
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=PdfPipelineOptions(do_ocr=False))}
)

pages = []
for pdf in sorted(Path("data").glob("*.pdf")):
    document = converter.convert(pdf).document
    for number in document.pages:
        pages.append({"source": pdf.name, "page": number, "text": document.export_to_markdown(page_no=number)})

print(len(pages), "pages")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

34 pages


Here is page 3 of the Schmidt paper as Docling sees it. The table came through as a table.

In [7]:
from IPython.display import Markdown

page = next(p for p in pages if p["source"].startswith("Schmidt") and p["page"] == 3)
Markdown(page["text"])

Figure 1. Principle of the multiplexed aptamer binding assay using VideoScan technology. (1) Dye/sizeencoded microbead populations presenting different proteins on their surfaces are mixed (2) incubated with a fluorescence-labelled aptamer (3) dissolved in a binding buffer of choice. After removing unbound aptamers by washing, the microbead suspension was transferred into a cavity of a 96 well plate (4). The microbeads were allowed to settle down forming a microbead chip on the transparent bottom (5). A fluorescence microscope (6) is used to take pictures of the microbead chip (7). Imaging software analyses the pictures, recognises and counts microbeads, measures their surface fluorescence intensity and groups them into populations (8). Finally, for each population the referenced mean fluorescence intensity (rMFI) per microbead population is calculated (9). (10) Shows a 96-well microtiter plate placed into the VideoScan system.

<!-- image -->

aptamers are used in combination. By performing analyses in a 96-well format and multiplexing microbeads, we were able to sample large quantities efficiently and in a cost effective manner.

## Results and discussion

Aptamer-target binding using a multiplex assay. Our experimental approach presented in Fig. 1 required dye/size-encoded microbead populations presenting potential aptamer target proteins and some nontarget proteins (as specificity controls) on their surfaces. We coupled different proteins via EDC-chemistry to carboxylated microbead populations and checked for successful coupling with appropriate detection probes (Supplementary Table S2). Results are shown in supplementary information (Supplementary Fig. S1).

Due to non-directional binding, the target proteins have random orientation on the surface. The orientation may have an influence on the binding (e.g., inaccessible binding site, deformation) or activity (e.g., catalysed activity) 13,22 . But we argue that the random orientation can also stochastically make accessible a portion that is usually sufficient for the generation of a measurement signal. This is also in line with our previous ­work 23-25 , where molecules were successfully bound.

We then examined the binding of the nine aptamers to microbeads presenting their target molecules (Fig. 2A). We were unable to detect any signal mIgG, although the presence of the corresponding target coupled to the surface of the microbeads was clearly shown by the use of antibodies (Supplementary Fig. S1). Aptamers are considered as a substitute for antibodies, and researchers often assume that aptamers can be modified in a similar

͹

Vol.:(0123456789)

A whole page is too much text to hand the model for every question, and too coarse to
find the one relevant paragraph. So we cut each page into **chunks** of about 1000
characters. Consecutive chunks overlap by 200 characters, so a sentence cut at a border
still appears whole in one of them.

![A page cut into three overlapping chunks](img/chunking.svg)

In [8]:
def chunk(text, size=1000, overlap=200):
    starts = range(0, max(len(text) - overlap, 1), size - overlap)
    return [text[start : start + size] for start in starts]


chunks = [
    {"source": p["source"], "page": p["page"], "text": piece}
    for p in pages
    for piece in chunk(p["text"])
    if piece.strip()
]
print(len(chunks), "chunks")
print(chunks[10]["text"])

197 chunks
om temperature (20 °C). The samples are continuously stirred during spectra and decay curve recording to prevent precipitation of beads. For steady state measurements, a xenon lamp has been used as excitation light source. Pulsed excitation for time-resolved measurements was provided by a Fianium Supercontinuum SC400-2-PP (NKT Photonics A/S) laser operated at a repetition rate of 10 MHz (for organic dyes) or 5 MHz (for QD-beads). In time-correlated single photon counting (TCSPC) experiments with an R38090U-50 (Hamamatsu Photonics K.K.) MCP-PMT, a pulse width of the instrument response function (IRF) of about 250 ps at 10 MHz was achieved.

Flow cytometer design. A new lifetime flow cytometer (LT-FCM) including a modulated laser source, novel single photon signal processing lifetime analysis electronics and signal analysis and instrument control software (CyPAD pantau) was designed (Quantum P/pantau, Quantum Analysis GmbH). The new signal processing for FCM is capable of sing

> **Look closely.** Where does this chunk start and end? Docling knows where every heading
> and table is, but our chunker ignores that and cuts every 800 characters, right through a
> table if one is in the way. The full template uses Docling's own chunker, which cuts along
> the document's structure instead.

## Part 3 · Embeddings and Qdrant

An **embedding** turns a text into a long list of numbers (a vector), so that texts with
similar *meaning* get similar vectors, even when they share no words.

![Two sentences about cats point the same way, a sentence about revenue points elsewhere](img/embeddings.svg)

In [9]:
import numpy as np


def embed(texts):
    response = llm.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def similarity(a, b):
    a, b = np.array(a), np.array(b)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))


cat, kitten, revenue = embed([
    "The cat sits on the mat.",
    "A kitten is resting on a rug.",
    "Quarterly revenue grew by four percent.",
])
print("numbers per vector:", len(cat))
print("cat vs kitten: ", round(similarity(cat, kitten), 3))
print("cat vs revenue:", round(similarity(cat, revenue), 3))

numbers per vector: 4096
cat vs kitten:  0.574
cat vs revenue: 0.133


Now embed every chunk. We send them in batches of 64; that's a handful of requests.

In [10]:
for start in range(0, len(chunks), 64):
    batch = chunks[start : start + 64]
    for item, vector in zip(batch, embed([c["text"] for c in batch])):
        item["vector"] = vector

print(len(chunks), "chunks embedded")

197 chunks embedded


Comparing a question against every chunk one by one works for 170 chunks, not for
millions. A **vector database** stores the vectors and finds the closest ones fast. We use
[Qdrant](https://qdrant.tech), the same database as the full template. `":memory:"` runs
it inside Python, with nothing to install. The template runs the same thing as a server.

A **collection** is like a table. Each **point** is one chunk: its vector, plus a
**payload** with everything we want back (text, file, page).

![A Qdrant collection: one point per chunk, each with id, vector and payload](img/qdrant.svg)

In [11]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)

hits = db.query_points("papers", query=embed([QUESTION])[0], limit=5).points
for hit in hits:
    print(f"{hit.score:.3f}  {hit.payload['source']} p. {hit.payload['page']}")
    print("      ", hit.payload["text"][:150].replace("\n", " "), "\n")

0.802  Kage_2018_SciReports.pdf p. 2
        extend the accessible lifetime range. The beads serve as a model system in our studies but further applications will not be limited to distinguishing 

0.747  Kage_2018_SciReports.pdf p. 4
       or lifetime encoding had to be excitable at a single common wavelength (here: 488 nm) and detectable within a certain spectral window. Moreover, they  

0.734  Kage_2018_SciReports.pdf p. 3
       Table 1. Lifetime codes and respective luminophores with measurement parameters, mean bead diameter D and obtained ensemble (reference) lifetimes calc 

0.733  Kage_2018_SciReports.pdf p. 2
       S core/shell quantum dots (QDs) and a final poly(sodium 4-styrensulfonate) (PSS) layer provided by the research group of Sukhanova and Nabiev 34,35 .  

0.710  Kage_2018_SciReports.pdf p. 4
       Figure 2. Photoluminescence decay curves of lifetime codes. (a) Spectroscopic ensemble measurements: normalised photoluminescence intensity decay curv 



The chatbot runs in a separate process, so it can't see the notebook's variables.
We save the chunks with their vectors to a file, and the app loads them into its own Qdrant.

In [12]:
import json

Path("index.json").write_text(json.dumps(chunks))
print("saved", len(chunks), "chunks to index.json")

saved 197 chunks to index.json


The bot becomes a search engine: no LLM for now, it just shows the best chunks.

In [13]:
%%writefile app.py
import json
import os

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

chunks = json.load(open("index.json"))
db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)


async def search(question, limit=5):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    return db.query_points("papers", query=response.data[0].embedding, limit=limit).points


@cl.on_message
async def on_message(message: cl.Message):
    hits = await search(message.content)
    results = [
        f"**{h.payload['source']}, p. {h.payload['page']}** (score {h.score:.2f})\n"
        f"> {h.payload['text'][:300].replace(chr(10), ' ')}…"
        for h in hits
    ]
    await cl.Message(content="\n\n".join(results)).send()

Overwriting app.py


Try a question in the browser, and also one the papers can't answer.
Search always returns *something*: the five closest chunks, however far away they are.

## Part 4 · Retrieval-augmented generation

Now we put the two halves together: **retrieve** the best chunks, **augment** the prompt
with them, and let the model **generate** an answer from that text. Here is the same
question as in Part 1, this time with context.

![The prompt: rules, the 5 numbered chunks and the question go to the LLM](img/rag-prompt.svg)

In [14]:
RAG_PROMPT = (
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say that you don't know."
)

context = "\n\n".join(f"[{i}] {hit.payload['text']}" for i, hit in enumerate(hits, start=1))

reply = llm.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": RAG_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {QUESTION}"},
    ],
)
print(reply.choices[0].message.content)

The lifetime-encoded beads were loaded with the following:
*   **Organic dyes** from PolyAn GmbH, Germany (used in PMMA beads).
*   **CdSe/CdS/ZnS core/shell quantum dots (QDs)** with a final poly(sodium 4-styrensulfonate) (PSS) layer (used in tailor-made melamine beads with a polyelectrolyte-based layer-by-layer coating).

The available lifetime codes range from slightly below 2 up to more than 20 ns.


Compare it with the answer from Part 1.

In the bot, the search runs inside a `cl.Step`, so you can unfold it in the chat and
see which chunks the answer was based on. Only the question and the answer go into the
history; the context is fetched fresh for each question.

In [15]:
%%writefile app.py
import json
import os

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
SYSTEM_PROMPT = (
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say that you don't know."
)

chunks = json.load(open("index.json"))
db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)


async def search(question, limit=5):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    return db.query_points("papers", query=response.data[0].embedding, limit=limit).points


@cl.on_chat_start
async def on_chat_start():
    cl.user_session.set("history", [{"role": "system", "content": SYSTEM_PROMPT}])


@cl.on_message
async def on_message(message: cl.Message):
    async with cl.Step(name="Search the papers", type="retrieval") as step:
        hits = await search(message.content)
        step.output = "\n\n".join(
            f"**{h.payload['source']}, p. {h.payload['page']}** ({h.score:.2f})" for h in hits
        )

    context = "\n\n".join(f"[{i}] {h.payload['text']}" for i, h in enumerate(hits, start=1))
    history = cl.user_session.get("history")
    prompt = {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message.content}"}

    answer = cl.Message(content="")
    stream = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"], messages=history + [prompt], stream=True
    )
    async for part in stream:
        if part.choices and part.choices[0].delta.content:
            await answer.stream_token(part.choices[0].delta.content)
    await answer.send()

    history.append({"role": "user", "content": message.content})
    history.append({"role": "assistant", "content": answer.content})

Overwriting app.py


## Part 5 · Citations

The chunks in the context are already numbered `[1]` to `[5]`. We ask the model to cite
those numbers, then look up which file and page each number stands for. A small regular
expression finds the numbers in the answer.

![The [1] in the answer leads to the first search hit, whose payload names the file and page](img/citations.svg)

In [16]:
import re

CITE_PROMPT = RAG_PROMPT + " After each statement, cite the context it comes from as [1], [2], …"

reply = llm.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": CITE_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {QUESTION}"},
    ],
)
text = reply.choices[0].message.content
print(text, "\n")

for number in sorted({int(n) for n in re.findall(r"\[(\d+)\]", text)}):
    if 1 <= number <= len(hits):
        print(f"[{number}] = {hits[number - 1].payload['source']}, page {hits[number - 1].payload['page']}")

The lifetime-encoded beads were loaded with:
*   Organic dyes from PolyAn GmbH, Germany [1].
*   CdSe/CdS/ZnS core/shell quantum dots (QDs) with a final poly(sodium 4-styrensulfonate) (PSS) layer [1].

The available lifetime codes range from slightly below 2 ns up to more than 20 ns [2]. Specific intensity-weighted mean lifetimes mentioned include:
*   Code A: 1.72 ns [3].
*   Code B: 2.71 ns [3].
*   Code C: 5.54 ns [3]. 

[1] = Kage_2018_SciReports.pdf, page 2
[2] = Kage_2018_SciReports.pdf, page 4
[3] = Kage_2018_SciReports.pdf, page 3


In the bot, each cited page becomes a `cl.Pdf` element. Chainlit turns the element's
name into a link wherever it appears in the message; clicking it opens the PDF at that page
in a side panel. This is the finished bot.

In [17]:
%%writefile app.py
import json
import os
import re

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
SYSTEM_PROMPT = (
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say that you don't know. "
    "After each statement, cite the context it comes from as [1], [2], …"
)

chunks = json.load(open("index.json"))
db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)


async def search(question, limit=5):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    return db.query_points("papers", query=response.data[0].embedding, limit=limit).points


@cl.on_chat_start
async def on_chat_start():
    cl.user_session.set("history", [{"role": "system", "content": SYSTEM_PROMPT}])


@cl.on_message
async def on_message(message: cl.Message):
    async with cl.Step(name="Search the papers", type="retrieval") as step:
        hits = await search(message.content)
        step.output = "\n\n".join(
            f"**{h.payload['source']}, p. {h.payload['page']}** ({h.score:.2f})" for h in hits
        )

    context = "\n\n".join(f"[{i}] {h.payload['text']}" for i, h in enumerate(hits, start=1))
    history = cl.user_session.get("history")
    prompt = {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message.content}"}

    answer = cl.Message(content="")
    stream = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"], messages=history + [prompt], stream=True
    )
    async for part in stream:
        if part.choices and part.choices[0].delta.content:
            await answer.stream_token(part.choices[0].delta.content)

    # Turn every cited [n] into a link that opens the PDF at the right page.
    pdfs = {}
    for number in sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer.content)}):
        if 1 <= number <= len(hits):
            source, page = hits[number - 1].payload["source"], hits[number - 1].payload["page"]
            name = f"{source.removesuffix('.pdf')}, p. {page}"
            pdfs[name] = cl.Pdf(name=name, path=f"data/{source}", page=page, display="side")
    if pdfs:
        answer.content += "\n\n**Sources:** " + " · ".join(pdfs)
        answer.elements = list(pdfs.values())
    await answer.send()

    history.append({"role": "user", "content": message.content})
    history.append({"role": "assistant", "content": answer.content})

Overwriting app.py


## Part 6 · Where to go from here

You've built every part of a RAG system: parsing, chunking, embedding, a vector database,
retrieval, generation and citations. The full template in `apps/chainlit/` does the same
steps, with production answers to the problems you saw today:

| Today | The template |
|---|---|
| PDFs only | Also txt, md, csv and json |
| Fixed 1000-character chunks | Docling's structure-aware chunker and others, chosen per data source in a config file |
| Qdrant in memory, rebuilt on every start | Qdrant server, ingested once, updated when files change |
| Vector search only | Hybrid search: vectors plus keyword matching |
| `[1]` and a file name | Citation format set in the config: title, file, page |
| You judge the answers by eye | An evaluation app that scores answers |

### Things to try

1. **Bring your own PDFs.** Put them in `data/`, re-run Parts 2 and 3. The bot picks up the new `index.json` once you re-run the last `%%writefile` cell.
2. **Chunk size.** Try `size=300` and `size=3000` in `chunk()`. Which questions get better, which get worse?
3. **`limit`.** Retrieve 2 or 15 chunks instead of 5. What happens to the answers?
4. **The system prompt.** Remove "say that you don't know", then ask about something the papers don't cover.
5. **Ask "What did Schmidt et al. study?"** The bot says it doesn't know. Open the search step: which pages came back, and why? (Hint: does any chunk say who wrote the paper?) How would you fix it? One idea: put each paper's title and authors at the start of every chunk before embedding.